<a href="https://colab.research.google.com/github/asnapareeth004-hue/encoder_decoder/blob/main/Copy_of_EncoderDecoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Translation with an LSTM Encoder-Decoder (PyTorch)

## Objective
Build a sequence-to-sequence (seq2seq) model that translates short English sentences into Hindi. The encoder reads the English sentence and compresses it into a context vector (the LSTM's final hidden and cell states). The decoder uses that context to generate the Hindi sentence one word at a time.

## Dataset
IITB English-Hindi Parallel Corpus (Hugging Face: `cfilt/iitb-english-hindi`), about 1.6 million English-Hindi sentence pairs. I use only short, deduplicated sentences (3-8 English words) so a plain LSTM can learn from them.

## Steps

**Part 1: Preparing the data**
1. Load the dataset and extract the sentence pairs
2. Separate the encoder input (English) and the decoder input (`<sos>` + Hindi)
3. Tokenize and vectorize both sides using vocabularies built from the data
4. Pad the sequences to equal length
5. Build the decoder output data (Hindi + `<eos>`, the decoder input shifted by one word)

**Part 2: Building and training the model**
1. Build the encoder layers (Embedding + LSTM)
2. Build the decoder layers (Embedding + LSTM + Linear)
3. Combine them into one Seq2Seq model
4. Train with teacher forcing and cross-entropy loss (padding ignored), checking validation loss each epoch

**Part 3: Inference**
Translate new sentences with greedy decoding: feed `<sos>`, pick the most likely word, feed it back in, and stop at `<eos>`.

## Notes
- Teacher forcing: during training the decoder receives the true previous word instead of its own prediction.
- Limitation: a plain LSTM encoder-decoder (no attention) squeezes the whole sentence into one fixed-size vector, so translation quality drops on longer or unusual sentences.

In [ ]:
# install and imports important packages
!pip install -q datasets

import re, random, torch, torch.nn as nn
from collections import Counter
from datasets import load_dataset
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

random.seed(0); torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"
print("device:", device)


device: cuda


In [ ]:
#download and extract sentence pairs
ds = load_dataset("cfilt/iitb-english-hindi", split="train")
print(ds)
print(ds[0])


Dataset({
    features: ['translation'],
    num_rows: 1659083
})
{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}


In [ ]:
# remove duplicates. Replace the pair-building loop
def clean_en(s): return re.sub(r"[^a-z\s]", " ", s.lower()).split()
def clean_hi(s): return re.sub(r"[^\u0900-\u0963\s]", " ", s).split()

pairs, seen = [], set()
sub = ds.shuffle(seed=0).select(range(400000))
for ex in sub:
    t = ex["translation"]
    e, h = clean_en(t["en"]), clean_hi(t["hi"])
    if 3 <= len(e) <= 8 and 3 <= len(h) <= 10 and tuple(e) not in seen:
        seen.add(tuple(e))
        pairs.append((e, h))
random.shuffle(pairs)
pairs = pairs[:60000]
print("pairs used:", len(pairs))

encoder_input_texts  = [e for e, _ in pairs]
decoder_input_texts  = [[SOS] + h for _, h in pairs]
decoder_target_texts = [h + [EOS] for _, h in pairs]
print(encoder_input_texts[0], decoder_input_texts[0], decoder_target_texts[0], sep="\n")

pairs used: 60000
['failed', 'to', 'download', 'messages', 'for', 'offline', 'viewing']
['<sos>', 'ऑफ़लाइन', 'प्रयोग', 'के', 'लिए', 'संदेश', 'डाउनलोड', 'करने', 'में', 'असफल']
['ऑफ़लाइन', 'प्रयोग', 'के', 'लिए', 'संदेश', 'डाउनलोड', 'करने', 'में', 'असफल', '<eos>']


In [ ]:
print(encoder_input_texts[10], decoder_input_texts[10], decoder_target_texts[10], sep="\n")

['state', 'industrial', 'investment', 'corporation']
['<sos>', 'राज्य', 'औद्योगिक', 'निवेश', 'निगम']
['राज्य', 'औद्योगिक', 'निवेश', 'निगम', '<eos>']


In [ ]:
#tokenize, vectorize, pad, DataLoader
def build_vocab(texts, max_words=8000):
    cnt = Counter(w for t in texts for w in t)
    itos = [PAD, SOS, EOS, UNK] + [w for w, _ in cnt.most_common(max_words)]
    return {w: i for i, w in enumerate(itos)}, itos

src_stoi, src_itos = build_vocab(encoder_input_texts)
tgt_stoi, tgt_itos = build_vocab([[SOS, EOS]] + decoder_input_texts + decoder_target_texts)

def vectorize(texts, stoi):
    return [torch.tensor([stoi.get(w, stoi[UNK]) for w in t]) for t in texts]

enc_seqs    = vectorize(encoder_input_texts, src_stoi)
dec_in_seqs = vectorize(decoder_input_texts, tgt_stoi)
dec_out_seqs= vectorize(decoder_target_texts, tgt_stoi)

enc_lens    = torch.tensor([len(s) for s in enc_seqs])
enc_data    = pad_sequence(enc_seqs,     batch_first=True, padding_value=0)
dec_in_data = pad_sequence(dec_in_seqs,  batch_first=True, padding_value=0)
dec_out_data= pad_sequence(dec_out_seqs, batch_first=True, padding_value=0)
print(enc_data.shape, dec_in_data.shape, dec_out_data.shape)
print("vocab sizes:", len(src_itos), len(tgt_itos))

loader = DataLoader(TensorDataset(enc_data, enc_lens, dec_in_data, dec_out_data),
                    batch_size=128, shuffle=True)

torch.Size([60000, 8]) torch.Size([60000, 11]) torch.Size([60000, 11])
vocab sizes: 8004 8004


In [ ]:
# build encoder, decoder, and model
EMB, HID = 256, 512

class Encoder(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.embedding = nn.Embedding(vocab, EMB, padding_idx=0)
        self.lstm = nn.LSTM(EMB, HID, batch_first=True)
    def forward(self, x, lengths):
        packed = pack_padded_sequence(self.embedding(x), lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (h, c) = self.lstm(packed)
        return h, c

class Decoder(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.embedding = nn.Embedding(vocab, EMB, padding_idx=0)
        self.lstm = nn.LSTM(EMB, HID, batch_first=True)
        self.fc = nn.Linear(HID, vocab)
    def forward(self, x, state):
        out, state = self.lstm(self.embedding(x), state)
        return self.fc(out), state

class Seq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab):
        super().__init__()
        self.encoder, self.decoder = Encoder(src_vocab), Decoder(tgt_vocab)
    def forward(self, src, src_len, dec_in):
        state = self.encoder(src, src_len)
        logits, _ = self.decoder(dec_in, state)
        return logits

model = Seq2Seq(len(src_itos), len(tgt_itos)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)
print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(8004, 256, padding_idx=0)
    (lstm): LSTM(256, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(8004, 256, padding_idx=0)
    (lstm): LSTM(256, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=8004, bias=True)
  )
)


In [ ]:
#Training the model
for epoch in range(1, 11):
    model.train(); total = 0
    for src, src_len, dec_in, dec_out in loader:
        src, dec_in, dec_out = src.to(device), dec_in.to(device), dec_out.to(device)
        logits = model(src, src_len, dec_in)
        loss = criterion(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); total += loss.item()
    print(f"epoch {epoch:2d}  loss {total / len(loader):.4f}")

epoch  1  loss 5.2526
epoch  2  loss 3.9691
epoch  3  loss 3.0823
epoch  4  loss 2.4119
epoch  5  loss 1.8836
epoch  6  loss 1.4528
epoch  7  loss 1.1027
epoch  8  loss 0.8229
epoch  9  loss 0.6054
epoch 10  loss 0.4419


In [ ]:
# translate and show output
def translate(sentence, max_len=15):
    model.eval()
    words = clean_en(sentence)
    x = torch.tensor([[src_stoi.get(w, src_stoi[UNK]) for w in words]]).to(device)
    with torch.no_grad():
        state = model.encoder(x, torch.tensor([len(words)]))
        tok, out = torch.tensor([[tgt_stoi[SOS]]]).to(device), []
        for _ in range(max_len):
            logits, state = model.decoder(tok, state)
            nxt = logits[:, -1].argmax(-1)
            if nxt.item() == tgt_stoi[EOS]: break
            out.append(tgt_itos[nxt.item()])
            tok = nxt.unsqueeze(1)
    return " ".join(out)

for s in ["how are you", "i am going home", "what is your name", "this is a good book"]:
    print(f"{s:25s} -> {translate(s)}")

how are you               -> तुम कैसे हो
i am going home           -> मैं घर जा रहा हूं
what is your name         -> नाम क्या है तुम्हारा
this is a good book       -> यह एक अच्छा पुस्तक है
